# Notebook 08: Temporal Analysis

**Pipeline layer I.** Recomputes the four-measure stockiness ensemble on time-binned subsets of the corpus and derives three temporal artifacts: per-element lifecycle, per-genre survival curves, and per-genre vocabulary change-points. Reads `element_index.parquet`, `movies_union.parquet`, and `stockiness_long_classified.parquet`. Closes the Restatement §18 mapping by addressing the "customary" descriptor that the per-cell measures in Notebook 03 cannot reach.

## Purpose

Notebook 03 scores stockiness against the full corpus pooled across decades. That construction answers four of the five Restatement of Copyright §18 descriptors: stock (Primary), common-but-distinctive (TF-IDF), stereotypical (Dunning G²), and standard (smoothed posterior). The fifth descriptor, **customary**, asks a question the per-cell measures cannot address: how has an element's conventionality evolved over time? An element that has been stock in a genre for eighty years and an element that became stock five years ago are both "stock" in 03's headline output, but they sit at very different points on the doctrine's customary axis. Doctrinally, customary conventions carry the strongest argument against protection: they are conventions every author of the genre has used for generations, embedded so deeply in the genre's vocabulary that no individual author can plausibly claim originality over them.

Notebook 08 operationalizes the customary descriptor by re-running 03's stockiness pipeline on time-binned subsets of the corpus. For each five-year bin from 1900 to 2020, the notebook recomputes the four measures and the ensemble using only films released in that bin. The result is a per-`(element, element_type, genre, bin)` table with the same schema as 03's per-cell table but indexed by time. From this per-bin table the notebook derives three temporal artifacts.

**Element lifecycle.** For each `(element, element_type, genre)` tuple in the framework's narrative-element vocabulary, trace the ensemble score across bins. Identify the bin of first appearance, the bin of peak stockiness, the bin of last appearance, the bins where the element first crossed and last crossed the "stock" threshold, and the duration of its tenure as stock. Smooth the trajectory with LOWESS so single-bin noise does not dominate the lifecycle estimates.

**Survival curves.** Given an element is at z_ensemble ≥ 1 in genre g at time τ, what is the probability it is still at z_ensemble ≥ 1 at time τ + δ? Aggregate across all τ for each δ ∈ {5, 10, 15, 20, 30, 50} years, per genre and corpus-wide. The resulting survival curves quantify "convention lifespan" — how long a genre convention typically persists once it is established.

**Vocabulary change-points.** For each genre and each consecutive pair of bins, compute the Jaccard similarity between the top-K stock elements at τ and the top-K at τ−1. Bins where the Jaccard is anomalously low (genre-specific bottom 10% threshold) are flagged as vocabulary change-points. These are the bins in which the genre's conventional vocabulary turned over substantially.

## Inputs

- `v4_rebuild/02_elements/element_index.parquet` — 5,658,755 element occurrences (one row per occurrence, with `union_id`, `element`, `element_type`)
- `v4_rebuild/01c_union/movies_union.parquet` — 45,958 films with `union_id`, `year`, `genres_canonical`
- `v4_rebuild/03b_classification/stockiness_long_classified.parquet` — the canonical narrative-element vocabulary (only `is_narrative_element=True` pairs are tracked)

## Outputs

- `v4_rebuild/08_temporal/stockiness_by_bin.parquet` — per-`(element, type, genre, bin)` long table with all four measures, z-scores, and the ensemble. The temporal analogue of `stockiness_long.parquet`.
- `v4_rebuild/08_temporal/element_lifecycle.parquet` — per-`(element, type, genre)` lifecycle summary: first bin, last bin, peak bin, peak ensemble, first stock crossing, last stock crossing, duration of stock tenure.
- `v4_rebuild/08_temporal/survival_curves.parquet` — per-genre and corpus-wide survival probabilities at each lag.
- `v4_rebuild/08_temporal/change_points.parquet` — per-genre vocabulary change-point bins with the Jaccard turnover signal.
- `v4_rebuild/08_temporal/temporal_report.txt` — headline temporal patterns, the top emerging and fading elements per genre, and the genre vocabulary-shift years.

## Cost and runtime

No API spend. No GPU. Pure pandas and scipy. Expected runtime 5-15 minutes, dominated by the per-bin sparse aggregations. Memory peak roughly 4-6 GB.

In [1]:
# ----------------------------------------------------------------------
# Setup and configuration
# ----------------------------------------------------------------------
!pip install -q statsmodels

import os, time
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# --- Google Drive -----------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')

# --- Paths ------------------------------------------------------------
ROOT = Path('/content/drive/MyDrive/DGK Lab/Reasoning Lab/'
            'DGK Lab - Scene A Faire Computational')
V4 = ROOT / 'Analysis/v4_rebuild'

PATH_ELEM_IDX = V4 / '02_elements'        / 'element_index.parquet'
PATH_MOVIES   = V4 / '01c_union'          / 'movies_union.parquet'
PATH_ENRICHED = V4 / '03b_classification' / 'stockiness_long_classified.parquet'

OUT_DIR = V4 / '08_temporal'
OUT_DIR.mkdir(parents=True, exist_ok=True)

PATH_BY_BIN     = OUT_DIR / 'stockiness_by_bin.parquet'
PATH_LIFECYCLE  = OUT_DIR / 'element_lifecycle.parquet'
PATH_SURVIVAL   = OUT_DIR / 'survival_curves.parquet'
PATH_CHANGE_PTS = OUT_DIR / 'change_points.parquet'
PATH_REPORT     = OUT_DIR / 'temporal_report.txt'

# --- Run configuration ------------------------------------------------
# Five-year bins from 1900 to 2020. The corpus spans 1895-2017, so the
# 1895-1899 films (a handful) are excluded; this is consistent with
# 03's filter conventions (rare bins produce unstable z-scores).
START_YEAR = 1900
END_YEAR   = 2020
BIN_SIZE   = 5

# Element filter per bin. Smaller than 03's corpus-wide filter (≥10
# instances AND ≥5 films) because per-bin populations are smaller; we
# loosen to ≥3 films per bin to keep early bins workable while still
# rejecting hapax noise.
MIN_FILMS_PER_BIN  = 3
MIN_INSTANCES_PER_BIN = 5

# Bin-level minimum film count. Bins with fewer than this many films
# produce unstable z-scores and are excluded from the cross-bin
# comparisons. Empirical check: corpus has roughly 100 films per
# five-year bin in 1900-1915 and thousands per bin in 1960-2010.
MIN_FILMS_PER_BIN_TOTAL = 100

# Stock threshold for survival analysis. An element is "stock" at time τ
# if its within-bin within-genre ensemble z-score is at least this value.
# 1.0 corresponds to roughly the top 16% of elements in the bin-genre.
STOCK_THRESHOLD_Z = 1.0

# Top-K for change-point computation. Per-genre top-K stock elements at
# each bin are compared bin-to-bin via Jaccard similarity. K=15 is the
# convention used elsewhere in the pipeline for headline rankings.
K_TOP = 15

# Survival lags in years
SURVIVAL_LAGS = [5, 10, 15, 20, 30, 50]

# Dirichlet smoothing alpha for the posterior measure (matches 03)
ALPHA_SMOOTH = 1.0

# Change-point threshold: per-genre, a bin is flagged as a change-point if
# its Jaccard with the previous bin\'s top-K is below the genre\'s 10th
# percentile across all consecutive-bin pairs. Genre-specific so genres
# with naturally faster turnover are not over-flagged.
CHANGE_POINT_PERCENTILE = 0.10

# Canonical genre list
CANONICAL_GENRES = [
    'Action', 'Adventure', 'Animation', 'Comedy', 'Crime',
    'Drama', 'Family', 'Fantasy', 'History', 'Horror',
    'Music', 'Mystery', 'Romance', 'Science Fiction',
    'Thriller', 'War', 'Western',
]
N_GENRES = len(CANONICAL_GENRES)

# Bin centers (start years), inclusive of START_YEAR, exclusive of END_YEAR.
# A film with year y belongs to bin (y - START_YEAR) // BIN_SIZE * BIN_SIZE + START_YEAR.
BIN_STARTS = list(range(START_YEAR, END_YEAR, BIN_SIZE))
N_BINS = len(BIN_STARTS)

print('Setup complete.')
print(f'  Bin range : {START_YEAR}-{END_YEAR}, BIN_SIZE = {BIN_SIZE} years, {N_BINS} bins')
print(f'  Filter    : ≥{MIN_INSTANCES_PER_BIN} instances AND ≥{MIN_FILMS_PER_BIN} films per (element, type, bin)')
print(f'  Min films/bin total: {MIN_FILMS_PER_BIN_TOTAL} (skip sparse early bins)')
print(f'  Stock z-score threshold: {STOCK_THRESHOLD_Z}')
print(f'  Top-K for change-point: {K_TOP}')
print(f'  Output dir: {OUT_DIR}')


Mounted at /content/drive
Setup complete.
  Bin range : 1900-2020, BIN_SIZE = 5 years, 24 bins
  Filter    : ≥5 instances AND ≥3 films per (element, type, bin)
  Min films/bin total: 100 (skip sparse early bins)
  Stock z-score threshold: 1.0
  Top-K for change-point: 15
  Output dir: /content/drive/MyDrive/DGK Lab/Reasoning Lab/DGK Lab - Scene A Faire Computational/Analysis/v4_rebuild/08_temporal


In [2]:
# ----------------------------------------------------------------------
# Load inputs and prepare the time-binned data
#
# Build three tables that drive every downstream step:
#   1. films_bin : (union_id, year, bin) for every film with year in
#      [START_YEAR, END_YEAR). Films outside the range are dropped.
#   2. narrative_pairs : the (element, element_type) vocabulary from 03b
#      filtered to is_narrative_element=True. This is the SAME 26,639
#      pairs the rest of the pipeline operates on, so 08\'s temporal
#      lifecycle is over the doctrinally meaningful subset.
#   3. ei_narr_bin : element_index restricted to narrative pairs, joined
#      with film bin and exploded by canonical genre. One row per
#      (occurrence × canonical genre) at the per-(film, bin, genre) level.
# ----------------------------------------------------------------------

print('Loading element_index...')
ei = pd.read_parquet(PATH_ELEM_IDX)
print(f'  {len(ei):,} element rows across {ei["union_id"].nunique():,} films')

print('Loading movies_union...')
mv = pd.read_parquet(PATH_MOVIES)
# parquet round-trip turns list columns into numpy arrays; coerce back
mv['genres_canonical'] = mv['genres_canonical'].apply(
    lambda gs: list(gs) if hasattr(gs, '__len__') and not isinstance(gs, str) else []
)
# Restrict to canonical genres only
mv['genres_canonical'] = mv['genres_canonical'].apply(
    lambda gs: [g for g in gs if g in CANONICAL_GENRES]
)
print(f'  {len(mv):,} films')

print('Loading stockiness_long_classified for narrative-element filter...')
stk_cls = pd.read_parquet(PATH_ENRICHED)
narrative_pairs = (stk_cls[stk_cls['is_narrative_element']]
                   [['element','element_type']]
                   .drop_duplicates()
                   .reset_index(drop=True))
print(f'  {len(narrative_pairs):,} narrative (element, element_type) pairs to track over time')

# Step 1: films_bin --- one row per film with a valid year in [START_YEAR, END_YEAR)
films_bin = mv[['union_id','year','genres_canonical']].copy()
films_bin['year'] = pd.to_numeric(films_bin['year'], errors='coerce')
films_bin = films_bin.dropna(subset=['year'])
films_bin = films_bin[(films_bin['year'] >= START_YEAR) & (films_bin['year'] < END_YEAR)].copy()
films_bin['year'] = films_bin['year'].astype(int)
films_bin['bin'] = ((films_bin['year'] - START_YEAR) // BIN_SIZE) * BIN_SIZE + START_YEAR

print(f'\nFilms with valid year in [{START_YEAR}, {END_YEAR}): {len(films_bin):,}')
print('Films per bin (raw, before filter):')
print(films_bin.groupby('bin').size().to_string())

# Apply the bin-level minimum film count
bin_counts = films_bin.groupby('bin').size()
usable_bins = sorted(bin_counts[bin_counts >= MIN_FILMS_PER_BIN_TOTAL].index.tolist())
films_bin = films_bin[films_bin['bin'].isin(usable_bins)].reset_index(drop=True)
print(f'\nUsable bins (≥{MIN_FILMS_PER_BIN_TOTAL} films): {len(usable_bins)} of {N_BINS}')
print(f'  range: {usable_bins[0]} to {usable_bins[-1]}')
print(f'  total usable films: {len(films_bin):,}')

# Step 2: ei_narr_bin --- one row per (element occurrence × canonical genre) tagged with bin
print('\nBuilding the per-(occurrence × genre × bin) table...')
ei_narr = ei.merge(narrative_pairs, on=['element','element_type'], how='inner')
print(f'  element_index restricted to narrative pairs: {len(ei_narr):,} rows (down from {len(ei):,})')

# Join with the film bin assignment (this also drops films outside the year range)
ei_narr_bin = ei_narr.merge(films_bin[['union_id','bin','genres_canonical']], on='union_id', how='inner')
print(f'  after joining film bin (drops out-of-range films): {len(ei_narr_bin):,} rows')

# Explode by canonical genre
ei_narr_bin = ei_narr_bin.explode('genres_canonical').rename(columns={'genres_canonical':'genre'})
ei_narr_bin = ei_narr_bin.dropna(subset=['genre'])
print(f'  after exploding by canonical genre: {len(ei_narr_bin):,} rows')

# Films-per-(bin, genre): the denominator for rel_freq_in_genre per bin
fbg = films_bin.explode('genres_canonical').rename(columns={'genres_canonical':'genre'})
fbg = fbg.dropna(subset=['genre'])
films_per_bin_genre = fbg.groupby(['bin','genre']).size().reset_index(name='N_bg')

# Total films per bin (denominator for some marginals)
N_per_bin = films_bin.groupby('bin').size().reset_index(name='N_bin')

print(f'\nPer-(bin, genre) film counts ready: {len(films_per_bin_genre):,} (bin, genre) cells')
print(f'Per-bin total film counts ready: {len(N_per_bin)} bins')


Loading element_index...
  5,658,714 element rows across 45,956 films
Loading movies_union...
  45,958 films
Loading stockiness_long_classified for narrative-element filter...
  28,066 narrative (element, element_type) pairs to track over time

Films with valid year in [1900, 2020): 44,153
Films per bin (raw, before filter):
bin
1900       8
1905      20
1910      93
1915     248
1920     318
1925     473
1930    1205
1935    1368
1940    1504
1945    1435
1950    1723
1955    1791
1960    1450
1965    1463
1970    1606
1975    1549
1980    1851
1985    2345
1990    2569
1995    3110
2000    4546
2005    7087
2010    4896
2015    1495

Usable bins (≥100 films): 21 of 24
  range: 1915 to 2015
  total usable films: 44,032

Building the per-(occurrence × genre × bin) table...
  element_index restricted to narrative pairs: 3,278,222 rows (down from 5,658,714)
  after joining film bin (drops out-of-range films): 3,179,870 rows
  after exploding by canonical genre: 7,753,017 rows

Per-(bin, 

In [3]:
# ----------------------------------------------------------------------
# Phase A — Per-bin recomputation of the four-measure stockiness ensemble
#
# For each time bin τ, recompute the four measures from Notebook 03
# using only films released in bin τ:
#
#   1. Primary Stockiness  = rel_freq_in_genre × distinctiveness
#                           = (f(e,g,τ)/N(g,τ)) × (f(e,g,τ)/f(e,τ))
#   2. Smoothed TF-IDF     = tf × (log((1+|G|)/(1+k(e,τ))) + 1)
#   3. Dunning G² (signed positive variant)
#   4. Posterior smoothed  = (f(e,g,τ) + α·P_prior(g,τ)) / (f(e,τ) + α)
#
# All within-bin within-genre z-scored (population SD, ddof=0). The
# ensemble is the equal-weighted average of the four z-scores, matching
# 03\'s convention exactly. The only difference from 03 is that every
# count is computed on the bin\'s film subset rather than the corpus.
#
# Implementation: aggregate all the counts in vectorized form first,
# then iterate over bins for the within-bin z-scoring. Z-scoring needs
# the within-bin within-genre population statistics, which are bin-
# specific, so iteration is unavoidable. Each bin\'s z-scoring is fast.
# ----------------------------------------------------------------------

# Step 1: All instance and film counts at once
print('Computing per-(element, type, genre, bin) counts...')

# Per-(element, type, genre, bin) instance count = row count after explode
eg_count = (ei_narr_bin.groupby(['element','element_type','genre','bin'])
                       .size()
                       .reset_index(name='instance_count'))

# Per-(element, type, genre, bin) film count = distinct films
eg_films = (ei_narr_bin[['element','element_type','genre','bin','union_id']]
            .drop_duplicates()
            .groupby(['element','element_type','genre','bin']).size()
            .reset_index(name='film_count'))

print(f'  (element, type, genre, bin) cells: {len(eg_count):,}')

# Per-(element, type, bin) marginals — computed PRE-explode by going back
# to the un-exploded table. This matches 03\'s convention.
print('Computing per-(element, type, bin) marginals...')
ei_narr_with_bin = ei_narr.merge(films_bin[['union_id','bin']], on='union_id', how='inner')

elem_total_bin = (ei_narr_with_bin.groupby(['element','element_type','bin'])
                                   .size()
                                   .reset_index(name='total_instance_count'))
elem_films_total_bin = (ei_narr_with_bin[['element','element_type','bin','union_id']]
                        .drop_duplicates()
                        .groupby(['element','element_type','bin']).size()
                        .reset_index(name='total_film_count'))
print(f'  per-(element, type, bin) marginals: {len(elem_total_bin):,} cells')

# Number of genres an element appears in within the bin
# (k(e, τ) in 03\'s notation)
elem_genre_span_bin = (eg_count.groupby(['element','element_type','bin'])['genre']
                                .nunique()
                                .reset_index(name='n_genres_present'))

# Step 2: Merge counts together and apply per-bin filter
print('\nMerging counts and applying per-bin filter...')
freq = (eg_count.merge(eg_films, on=['element','element_type','genre','bin'])
                .merge(elem_total_bin, on=['element','element_type','bin'])
                .merge(elem_films_total_bin, on=['element','element_type','bin'])
                .merge(elem_genre_span_bin, on=['element','element_type','bin']))

# Filter: ≥MIN_INSTANCES_PER_BIN instances AND ≥MIN_FILMS_PER_BIN films
# for the (element, type) within the bin
before_filter = len(freq)
freq = freq[
    (freq['total_instance_count'] >= MIN_INSTANCES_PER_BIN) &
    (freq['total_film_count']     >= MIN_FILMS_PER_BIN)
].reset_index(drop=True)
print(f'  cells before filter: {before_filter:,}')
print(f'  cells after  filter: {len(freq):,}')

# Step 3: Merge per-(bin, genre) and per-bin film totals
freq = (freq.merge(films_per_bin_genre, on=['bin','genre'])
            .merge(N_per_bin, on='bin'))

# Step 4: Compute the four measures, vectorized

# Primary Stockiness
freq['rel_freq_in_genre'] = freq['film_count']  / freq['N_bg']
freq['distinctiveness']   = freq['film_count']  / freq['total_film_count']
freq['primary_stockiness']= freq['rel_freq_in_genre'] * freq['distinctiveness']

# Smoothed TF-IDF — needs the per-(genre, bin) total instances as
# the TF denominator
genre_total_inst_bin = freq.groupby(['bin','genre'])['instance_count'].sum().reset_index(name='genre_total_inst_bin')
freq = freq.merge(genre_total_inst_bin, on=['bin','genre'])
freq['tf'] = freq['instance_count'] / freq['genre_total_inst_bin']
freq['idf'] = np.log((1 + N_GENRES) / (1 + freq['n_genres_present'])) + 1
freq['tfidf'] = freq['tf'] * freq['idf']

# Dunning G² (positive variant): 2x2 contingency log-likelihood
# a = c(e, g, τ); b = c(e, τ) - a; c = N_g(τ) - a; d = N(τ) - N_g(τ) - b
# (using instance counts; the genre marginal is genre_total_inst_bin)
def compute_dunning_g2_per_bin(df):
    """Vectorized signed G² over the bin. df has instance_count,
    total_instance_count, genre_total_inst_bin; we compute a, b, c, d
    against the within-bin total instance count."""
    a = df['instance_count'].values.astype(float)
    e_total = df['total_instance_count'].values.astype(float)
    g_total = df['genre_total_inst_bin'].values.astype(float)
    # Within-bin total instance count = sum of instance_count across the bin
    # (same for every row of the bin, but we recompute here for clarity)
    # Actually let\'s compute N once per bin then attach
    N_bin_inst = df.groupby('bin')['instance_count'].transform('sum').values.astype(float)

    b = e_total - a
    c = g_total - a
    d = N_bin_inst - g_total - b

    row1 = a + b; row2 = c + d
    col1 = a + c; col2 = b + d
    N    = row1 + row2

    e_a = row1 * col1 / N
    e_b = row1 * col2 / N
    e_c = row2 * col1 / N
    e_d = row2 * col2 / N

    def safe_term(o, e):
        with np.errstate(divide='ignore', invalid='ignore'):
            t = o * np.log(o / e)
        return np.where(o > 0, t, 0.0)

    g2 = 2 * (safe_term(a, e_a) + safe_term(b, e_b) + safe_term(c, e_c) + safe_term(d, e_d))
    sign = np.where(a > e_a, 1.0, -1.0)
    return sign * g2

freq['dunning_g2'] = compute_dunning_g2_per_bin(freq)
freq['dunning_g2_positive'] = freq['dunning_g2'].clip(lower=0)

# Posterior smoothed — Dirichlet (α=1) with the corpus-wide bin-genre prior
# P_prior(g, τ) = N(g, τ) / Σ_{g\'} N(g\', τ). The (bin, genre) prior
# is computed once and joined.
prior = films_per_bin_genre.copy()
prior['sum_per_bin'] = prior.groupby('bin')['N_bg'].transform('sum')
prior['p_prior_g']    = prior['N_bg'] / prior['sum_per_bin']
freq = freq.merge(prior[['bin','genre','p_prior_g']], on=['bin','genre'])

freq['posterior_smoothed'] = (
    (freq['film_count'] + ALPHA_SMOOTH * freq['p_prior_g']) /
    (freq['total_film_count'] + ALPHA_SMOOTH)
)

# Step 5: Within-(bin, genre) z-scores for the four ensemble members
print('\nComputing within-(bin, genre) z-scores...')

def zscore_population(s):
    mu = s.mean()
    sd = s.std(ddof=0)
    if sd == 0 or np.isnan(sd):
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - mu) / sd

for measure in ['primary_stockiness','tfidf','dunning_g2_positive','posterior_smoothed']:
    freq[f'{measure}_z'] = freq.groupby(['bin','genre'])[measure].transform(zscore_population)

# Ensemble = mean of the four z-scores
freq['ensemble'] = (
    freq['primary_stockiness_z']
    + freq['tfidf_z']
    + freq['dunning_g2_positive_z']
    + freq['posterior_smoothed_z']
) / 4

print(f'  per-(element, type, genre, bin) ensemble cells: {len(freq):,}')
print(f'  ensemble range: [{freq["ensemble"].min():.3f}, {freq["ensemble"].max():.3f}]')
print(f'  ensemble per-(bin, genre) mean (should be ≈ 0): '
      f'{freq.groupby(["bin","genre"])["ensemble"].mean().abs().max():.6f}')

# Save the per-bin table
out_cols = [
    'element','element_type','genre','bin',
    'instance_count','film_count',
    'total_instance_count','total_film_count','n_genres_present',
    'N_bg','N_bin','p_prior_g',
    'rel_freq_in_genre','distinctiveness',
    'primary_stockiness','tf','idf','tfidf',
    'dunning_g2','dunning_g2_positive','posterior_smoothed',
    'primary_stockiness_z','tfidf_z','dunning_g2_positive_z','posterior_smoothed_z',
    'ensemble',
]
freq[out_cols].to_parquet(PATH_BY_BIN, index=False)
print(f'\nSaved: {PATH_BY_BIN}  ({os.path.getsize(PATH_BY_BIN)/1e6:.1f} MB, {len(freq):,} rows)')


Computing per-(element, type, genre, bin) counts...
  (element, type, genre, bin) cells: 1,431,481
Computing per-(element, type, bin) marginals...
  per-(element, type, bin) marginals: 265,244 cells

Merging counts and applying per-bin filter...
  cells before filter: 1,431,481
  cells after  filter: 815,444

Computing within-(bin, genre) z-scores...
  per-(element, type, genre, bin) ensemble cells: 815,444
  ensemble range: [-1.256, 34.523]
  ensemble per-(bin, genre) mean (should be ≈ 0): 0.000000

Saved: /content/drive/MyDrive/DGK Lab/Reasoning Lab/DGK Lab - Scene A Faire Computational/Analysis/v4_rebuild/08_temporal/stockiness_by_bin.parquet  (57.8 MB, 815,444 rows)


In [4]:
# ----------------------------------------------------------------------
# Phase B — Element lifecycle
#
# For each (element, element_type, genre), trace the ensemble z-score
# across bins and derive lifecycle summaries:
#
#   first_bin           — earliest bin where the element has any presence
#                          (instance_count >= 1)
#   last_bin            — latest bin with any presence
#   peak_bin            — bin of maximum smoothed ensemble z-score
#   peak_ensemble       — the smoothed ensemble at peak_bin
#   first_stock_bin     — first bin where smoothed ensemble crossed
#                          STOCK_THRESHOLD_Z (=1.0)
#   last_stock_bin      — last bin where smoothed ensemble was above
#                          STOCK_THRESHOLD_Z
#   stock_duration_bins — last_stock_bin - first_stock_bin (in bins, not years)
#   ever_stock          — True if the element was ever above threshold
#
# Smoothing: LOWESS with frac=0.3 over the per-bin ensemble trajectory.
# This dampens single-bin noise (a year where an element happened to
# appear in three unusual films) without flattening genuine lifecycle
# inflection points.
# ----------------------------------------------------------------------

from statsmodels.nonparametric.smoothers_lowess import lowess

print('Building element lifecycle table...')

# Each unique (element, type, genre) gets a row.
# We need the trajectory of ensemble across all usable bins, with
# missing bins represented as NaN so the smoother handles them.
keys = freq[['element','element_type','genre']].drop_duplicates().reset_index(drop=True)
print(f'  unique (element, type, genre) tuples: {len(keys):,}')

bin_array = np.array(usable_bins, dtype=int)
n_bins = len(bin_array)

# Pivot the freq table to wide form (element-type-genre × bin → ensemble)
# This is the most efficient way to access per-key trajectories.
print('  pivoting to wide form...')
freq['key'] = list(zip(freq['element'], freq['element_type'], freq['genre']))
wide = freq.pivot_table(index='key', columns='bin', values='ensemble', aggfunc='first')
# Add any missing bin columns (bins in usable_bins but absent from data)
for b in bin_array:
    if b not in wide.columns:
        wide[b] = np.nan
wide = wide.reindex(columns=bin_array)
print(f'  wide table shape: {wide.shape}')

# Smooth each row\'s trajectory with LOWESS. Missing bins are treated as
# NaN; LOWESS skips them and smooths over the present points. The
# smoothed value at a missing bin is interpolated from the smoother.
print('  smoothing trajectories with LOWESS (frac=0.3)...')
smoothed = np.empty_like(wide.values, dtype=float)
smoothed[:] = np.nan

frac = 0.3
for row_i in tqdm(range(len(wide)), desc='lifecycle smoothing'):
    y = wide.values[row_i, :]
    mask = ~np.isnan(y)
    if mask.sum() < 3:
        # Too few points for LOWESS; copy raw values
        smoothed[row_i, :] = y
        continue
    # LOWESS needs x-values; use bin start years
    xs = bin_array[mask]
    ys = y[mask]
    result = lowess(ys, xs, frac=frac, return_sorted=True)
    # Interpolate the smoothed curve back to all usable bins
    smoothed_vals = np.interp(bin_array, result[:, 0], result[:, 1],
                              left=np.nan, right=np.nan)
    # At bins where the raw was NaN AND outside the support of LOWESS, leave NaN
    smoothed[row_i, :] = smoothed_vals

# Build the lifecycle records
print('  deriving lifecycle summaries...')
lifecycle_rows = []
for row_i, key in enumerate(wide.index):
    element, element_type, genre = key
    y_raw = wide.values[row_i, :]
    y_sm  = smoothed[row_i, :]

    # First/last bin with any presence (raw)
    present = ~np.isnan(y_raw)
    if present.sum() == 0:
        continue
    first_bin = bin_array[present][0]
    last_bin  = bin_array[present][-1]

    # Peak bin via smoothed values
    if np.all(np.isnan(y_sm)):
        peak_bin = int(first_bin)
        peak_ensemble = np.nan
    else:
        peak_idx = np.nanargmax(y_sm)
        peak_bin = int(bin_array[peak_idx])
        peak_ensemble = float(y_sm[peak_idx])

    # Stock crossings via smoothed values
    stock_mask = y_sm >= STOCK_THRESHOLD_Z
    if stock_mask.any():
        first_stock_bin = int(bin_array[stock_mask][0])
        last_stock_bin  = int(bin_array[stock_mask][-1])
        stock_duration_bins = (last_stock_bin - first_stock_bin) // BIN_SIZE + 1
        ever_stock = True
    else:
        first_stock_bin = None
        last_stock_bin  = None
        stock_duration_bins = 0
        ever_stock = False

    lifecycle_rows.append({
        'element': element,
        'element_type': element_type,
        'genre': genre,
        'first_bin': int(first_bin),
        'last_bin':  int(last_bin),
        'peak_bin':  peak_bin,
        'peak_ensemble': peak_ensemble,
        'first_stock_bin': first_stock_bin,
        'last_stock_bin':  last_stock_bin,
        'stock_duration_bins': stock_duration_bins,
        'ever_stock': ever_stock,
        'n_bins_present': int(present.sum()),
    })

lifecycle = pd.DataFrame(lifecycle_rows)
lifecycle.to_parquet(PATH_LIFECYCLE, index=False)
print(f'\nElement lifecycle table: {len(lifecycle):,} (element, type, genre) tuples')
print(f'  ever stock: {int(lifecycle["ever_stock"].sum()):,}  ({lifecycle["ever_stock"].mean()*100:.1f}%)')
print(f'  stock duration (bins) distribution among ever-stock:')
ever_stock = lifecycle[lifecycle['ever_stock']]
print(f'    median = {ever_stock["stock_duration_bins"].median()}')
print(f'    mean   = {ever_stock["stock_duration_bins"].mean():.1f}')
print(f'    max    = {ever_stock["stock_duration_bins"].max()}')
print(f'Saved: {PATH_LIFECYCLE}')


Building element lifecycle table...
  unique (element, type, genre) tuples: 143,691
  pivoting to wide form...
  wide table shape: (143691, 21)
  smoothing trajectories with LOWESS (frac=0.3)...


lifecycle smoothing:   0%|          | 0/143691 [00:00<?, ?it/s]

  deriving lifecycle summaries...

Element lifecycle table: 143,691 (element, type, genre) tuples
  ever stock: 15,219  (10.6%)
  stock duration (bins) distribution among ever-stock:
    median = 2.0
    mean   = 4.4
    max    = 21
Saved: /content/drive/MyDrive/DGK Lab/Reasoning Lab/DGK Lab - Scene A Faire Computational/Analysis/v4_rebuild/08_temporal/element_lifecycle.parquet


In [5]:
# ----------------------------------------------------------------------
# Phase C — Survival curves
#
# Question: given an element is at z_ensemble ≥ STOCK_THRESHOLD_Z in
# genre g at time τ, what is the probability it is still at z ≥ threshold
# at time τ + δ?
#
# Compute for each lag δ ∈ SURVIVAL_LAGS (years). The lag is converted
# to bins via δ // BIN_SIZE. For each lag, iterate over all (τ, τ+δ)
# bin pairs that both lie within the usable range, and aggregate over
# all (element, type, g) tuples that have ensemble values at both bins.
#
# Reported quantities:
#   - per-genre survival probability per lag
#   - corpus-wide survival probability per lag (averaged over genres)
#   - n_observations per lag (the denominator)
#
# Doctrinally: long survival = conventions are durable; short survival
# = conventions turn over quickly. The "customary" descriptor is most
# strongly engaged by conventions with long survival curves.
# ----------------------------------------------------------------------

print('Computing survival curves...')

# Reshape freq into a per-(key, bin) lookup
# We already have `wide` from Cell 4; reuse it.
key_arr = wide.index.values  # array of (element, type, genre) tuples
ensemble_arr = wide.values   # shape (n_keys, n_bins), NaN where missing
bin_idx_of = {b: i for i, b in enumerate(bin_array)}

# Build per-key genre lookup for the per-genre aggregation
key_genre = np.array([k[2] for k in key_arr])

survival_rows = []
for lag_years in SURVIVAL_LAGS:
    lag_bins = lag_years // BIN_SIZE  # integer number of bins
    if lag_bins == 0:
        continue
    # For each (τ, τ+lag) pair within usable bins
    n_pairs = max(0, len(bin_array) - lag_bins)
    if n_pairs == 0:
        continue

    # Stock mask at each bin: ensemble >= STOCK_THRESHOLD_Z
    stock_mask = ensemble_arr >= STOCK_THRESHOLD_Z

    # For corpus-wide and per-genre survival:
    # Numerator   = number of (key, τ) pairs where stock at τ AND stock at τ+lag
    # Denominator = number of (key, τ) pairs where stock at τ AND both bins have a value

    # Stock at both bins (numerator): stock_mask[:, :n_pairs] AND stock_mask[:, lag_bins:]
    stock_at_tau     = stock_mask[:, :n_pairs]
    stock_at_tau_plus = stock_mask[:, lag_bins : lag_bins + n_pairs]
    has_value_at_tau_plus = ~np.isnan(ensemble_arr[:, lag_bins : lag_bins + n_pairs])

    # Corpus-wide
    n_stock_at_tau_total = stock_at_tau.sum()
    # Restrict to pairs where the future bin also has a value (so we can verify stock status)
    valid_pairs = stock_at_tau & has_value_at_tau_plus
    survived = valid_pairs & stock_at_tau_plus
    n_valid_total = valid_pairs.sum()
    n_survived_total = survived.sum()
    survival_prob = n_survived_total / n_valid_total if n_valid_total > 0 else np.nan
    survival_rows.append({
        'lag_years': lag_years, 'lag_bins': int(lag_bins),
        'genre': 'CORPUS',
        'n_pairs': int(n_valid_total),
        'n_survived': int(n_survived_total),
        'survival_probability': float(survival_prob),
    })

    # Per-genre
    for g in CANONICAL_GENRES:
        mask_g = (key_genre == g)
        valid_g = valid_pairs[mask_g]
        survived_g = survived[mask_g]
        n_valid_g = valid_g.sum()
        n_survived_g = survived_g.sum()
        prob_g = n_survived_g / n_valid_g if n_valid_g > 0 else np.nan
        survival_rows.append({
            'lag_years': lag_years, 'lag_bins': int(lag_bins),
            'genre': g,
            'n_pairs': int(n_valid_g),
            'n_survived': int(n_survived_g),
            'survival_probability': float(prob_g),
        })

survival_df = pd.DataFrame(survival_rows)
survival_df.to_parquet(PATH_SURVIVAL, index=False)

print(f'\nCorpus-wide survival probabilities:')
corpus = survival_df[survival_df['genre']=='CORPUS'].sort_values('lag_years')
for _, r in corpus.iterrows():
    print(f'  lag = {int(r["lag_years"]):>2} years: P(stock | stock at τ) = {r["survival_probability"]:.3f}  '
          f'(n_pairs = {int(r["n_pairs"]):,})')

print(f'\nPer-genre survival at lag=20 years:')
at_20 = survival_df[(survival_df['lag_years']==20) & (survival_df['genre']!='CORPUS')]
for _, r in at_20.sort_values('survival_probability', ascending=False).iterrows():
    print(f'  {r["genre"]:<18} P = {r["survival_probability"]:.3f}  (n_pairs = {int(r["n_pairs"]):,})')
print(f'Saved: {PATH_SURVIVAL}')


Computing survival curves...

Corpus-wide survival probabilities:
  lag =  5 years: P(stock | stock at τ) = 0.568  (n_pairs = 44,035)
  lag = 10 years: P(stock | stock at τ) = 0.547  (n_pairs = 40,677)
  lag = 15 years: P(stock | stock at τ) = 0.538  (n_pairs = 36,433)
  lag = 20 years: P(stock | stock at τ) = 0.535  (n_pairs = 32,540)
  lag = 30 years: P(stock | stock at τ) = 0.517  (n_pairs = 26,161)
  lag = 50 years: P(stock | stock at τ) = 0.492  (n_pairs = 16,373)

Per-genre survival at lag=20 years:
  Drama              P = 0.732  (n_pairs = 2,542)
  Science Fiction    P = 0.595  (n_pairs = 1,320)
  Action             P = 0.568  (n_pairs = 2,532)
  Adventure          P = 0.567  (n_pairs = 2,750)
  Crime              P = 0.564  (n_pairs = 2,168)
  Comedy             P = 0.555  (n_pairs = 3,267)
  Thriller           P = 0.548  (n_pairs = 2,304)
  Horror             P = 0.548  (n_pairs = 1,781)
  Romance            P = 0.519  (n_pairs = 2,081)
  War                P = 0.518  (n_pair

In [6]:
# ----------------------------------------------------------------------
# Phase D — Vocabulary change-points
#
# For each genre and each consecutive pair of bins (τ-1, τ):
#   - top_K(g, τ-1) = top-K (element, type) pairs by ensemble in genre g
#                     at bin τ-1 (K = K_TOP)
#   - top_K(g, τ)   = top-K at bin τ
#   - jaccard(g, τ) = |top_K(τ-1) ∩ top_K(τ)| / |top_K(τ-1) ∪ top_K(τ)|
#
# A genre\'s vocabulary "shifts" at bin τ when jaccard(g, τ) is in
# the bottom CHANGE_POINT_PERCENTILE of the genre\'s consecutive-bin
# Jaccards. The percentile is genre-specific so that genres with
# naturally faster turnover are not over-flagged.
#
# Output: per-(genre, bin) Jaccard plus a boolean flag for change-point.
# ----------------------------------------------------------------------

print('Computing per-genre top-K and consecutive-bin Jaccards...')

# Per-(genre, bin) top-K stock elements by ensemble
top_k_rows = []
for (g, b), grp in freq.groupby(['genre','bin']):
    topk = grp.nlargest(K_TOP, 'ensemble')[['element','element_type','ensemble']]
    top_k_rows.append({
        'genre': g,
        'bin': b,
        'top_k_keys': set(zip(topk['element'], topk['element_type'])),
        'top_k_size': len(topk),
    })
top_k_df = pd.DataFrame(top_k_rows)

# Consecutive-bin Jaccards per genre
change_rows = []
for g in CANONICAL_GENRES:
    sub = top_k_df[top_k_df['genre']==g].sort_values('bin').reset_index(drop=True)
    if len(sub) < 2:
        continue
    for i in range(1, len(sub)):
        prev_set = sub['top_k_keys'].iloc[i-1]
        curr_set = sub['top_k_keys'].iloc[i]
        if len(prev_set) == 0 or len(curr_set) == 0:
            jacc = np.nan
        else:
            jacc = len(prev_set & curr_set) / len(prev_set | curr_set)
        change_rows.append({
            'genre': g,
            'from_bin': int(sub['bin'].iloc[i-1]),
            'to_bin':   int(sub['bin'].iloc[i]),
            'jaccard':  float(jacc) if not np.isnan(jacc) else np.nan,
            'top_k_size_from': int(sub['top_k_size'].iloc[i-1]),
            'top_k_size_to':   int(sub['top_k_size'].iloc[i]),
        })
change_df = pd.DataFrame(change_rows)

# Per-genre change-point flag: Jaccard below the genre\'s bottom percentile
print('\nFlagging change-points (per-genre bottom percentile)...')
change_df['is_change_point'] = False
for g in CANONICAL_GENRES:
    mask = (change_df['genre']==g) & change_df['jaccard'].notna()
    if mask.sum() < 3:
        continue
    threshold = change_df.loc[mask, 'jaccard'].quantile(CHANGE_POINT_PERCENTILE)
    change_df.loc[mask & (change_df['jaccard'] <= threshold), 'is_change_point'] = True

change_df.to_parquet(PATH_CHANGE_PTS, index=False)
print(f'\nChange-point summary by genre:')
for g in CANONICAL_GENRES:
    sub = change_df[change_df['genre']==g]
    if len(sub) == 0:
        continue
    cps = sub[sub['is_change_point']]
    median_jacc = sub['jaccard'].median()
    if len(cps):
        cp_bins = sorted(cps['to_bin'].tolist())
        cp_str = ', '.join(str(b) for b in cp_bins)
    else:
        cp_str = '(none flagged)'
    print(f'  {g:<18} median jaccard = {median_jacc:.3f}, change-points at bins: {cp_str}')
print(f'\nSaved: {PATH_CHANGE_PTS}')


Computing per-genre top-K and consecutive-bin Jaccards...

Flagging change-points (per-genre bottom percentile)...

Change-point summary by genre:
  Action             median jaccard = 0.252, change-points at bins: 1920, 1930
  Adventure          median jaccard = 0.304, change-points at bins: 1920, 1930
  Animation          median jaccard = 0.200, change-points at bins: 1920, 1930, 1980
  Comedy             median jaccard = 0.429, change-points at bins: 1920, 1925, 1975
  Crime              median jaccard = 0.429, change-points at bins: 1920, 1925
  Drama              median jaccard = 0.716, change-points at bins: 1920, 1930, 1940, 1975, 2015
  Family             median jaccard = 0.200, change-points at bins: 1925, 1930
  Fantasy            median jaccard = 0.111, change-points at bins: 1925, 1945, 1975
  History            median jaccard = 0.071, change-points at bins: 1930, 1935, 1955
  Horror             median jaccard = 0.334, change-points at bins: 1920, 1925
  Music              

In [7]:
# ----------------------------------------------------------------------
# Phase E — Human-readable report
#
# Layout:
#   - Setup summary
#   - Headline temporal patterns (corpus-wide survival, lifecycle)
#   - Top "emerging" elements per genre (largest peak_bin - first_bin gap)
#   - Top "fading" elements per genre (peak before 2000, last_bin earlier)
#   - "Persistent" conventions per genre (longest stock_duration_bins)
#   - Per-genre vocabulary change-point bins
# ----------------------------------------------------------------------
lines = []
def out(line=''):
    print(line); lines.append(line)

out('=' * 70)
out('NOTEBOOK 08: TEMPORAL ANALYSIS REPORT')
out('=' * 70)
out(f'  bin width                 : {BIN_SIZE} years')
out(f'  range covered             : {usable_bins[0]} to {usable_bins[-1]} ({len(usable_bins)} bins)')
out(f'  films per bin (min/max)   : {films_bin.groupby("bin").size().min()} / {films_bin.groupby("bin").size().max()}')
out(f'  narrative pairs tracked   : {len(narrative_pairs):,}')
out(f'  per-(element, type, genre, bin) cells after filter: {len(freq):,}')
out(f'  per-(element, type, genre) lifecycle rows: {len(lifecycle):,}')
out(f'  ever-stock rate           : {lifecycle["ever_stock"].mean()*100:.1f}%')
out('')

out('=' * 70)
out('HEADLINE: CORPUS-WIDE SURVIVAL CURVE')
out('=' * 70)
out('Given an element is at z_ensemble ≥ {} in some genre at time τ,'.format(STOCK_THRESHOLD_Z))
out('what is the probability it is still above the threshold at time τ + lag?')
out('')
corpus = survival_df[survival_df['genre']=='CORPUS'].sort_values('lag_years')
out(f'  {"lag (years)":>12} {"P(stock | stock at τ)":>22} {"n_pairs":>12}')
for _, r in corpus.iterrows():
    out(f'  {int(r["lag_years"]):>12} {r["survival_probability"]:>22.3f} {int(r["n_pairs"]):>12,}')
out('')

out('=' * 70)
out('PER-GENRE SURVIVAL AT LAG = 20 YEARS')
out('=' * 70)
at_20 = survival_df[(survival_df['lag_years']==20) & (survival_df['genre']!='CORPUS')]
out('(higher = genre\'s conventions are more durable)')
for _, r in at_20.sort_values('survival_probability', ascending=False).iterrows():
    out(f'  {r["genre"]:<18} P = {r["survival_probability"]:.3f}  (n_pairs = {int(r["n_pairs"]):,})')
out('')

out('=' * 70)
out('PER-GENRE PERSISTENT CONVENTIONS (longest stock duration in bins)')
out('=' * 70)
ever_stock = lifecycle[lifecycle['ever_stock']]
for g in CANONICAL_GENRES:
    sub = ever_stock[ever_stock['genre']==g].nlargest(8, 'stock_duration_bins')
    if len(sub) == 0:
        continue
    items = ', '.join(f'{r["element"]} ({int(r["stock_duration_bins"])} bins)' for _, r in sub.iterrows())
    out(f'  {g:<18} {items}')
out('')

out('=' * 70)
out('PER-GENRE EMERGING CONVENTIONS (first stock bin in 1985+; ever-stock)')
out('=' * 70)
EMERGE_CUT = 1985
recent_emergers = ever_stock[ever_stock['first_stock_bin'].fillna(0) >= EMERGE_CUT]
for g in CANONICAL_GENRES:
    sub = (recent_emergers[recent_emergers['genre']==g]
           .sort_values('first_stock_bin', ascending=False)
           .head(8))
    if len(sub) == 0:
        continue
    items = ', '.join(f'{r["element"]} (first stock {int(r["first_stock_bin"])})' for _, r in sub.iterrows())
    out(f'  {g:<18} {items}')
out('')

out('=' * 70)
out('PER-GENRE FADING CONVENTIONS (peak before 1990; last_stock_bin < 2000)')
out('=' * 70)
faders = ever_stock[(ever_stock['peak_bin'] < 1990) & (ever_stock['last_stock_bin'].fillna(0) < 2000)]
for g in CANONICAL_GENRES:
    sub = (faders[faders['genre']==g]
           .sort_values('last_stock_bin')
           .head(8))
    if len(sub) == 0:
        continue
    items = ', '.join(f'{r["element"]} (peak {int(r["peak_bin"])}, last stock {int(r["last_stock_bin"])})'
                      for _, r in sub.iterrows())
    out(f'  {g:<18} {items}')
out('')

out('=' * 70)
out('PER-GENRE VOCABULARY CHANGE-POINTS (bins where top-K turnover is in bottom 10%)')
out('=' * 70)
for g in CANONICAL_GENRES:
    sub = change_df[(change_df['genre']==g) & change_df['is_change_point']].sort_values('to_bin')
    if len(sub) == 0:
        out(f'  {g:<18} (no change-points flagged)')
        continue
    items = ', '.join(f'{int(r["to_bin"])} (jacc={r["jaccard"]:.2f})' for _, r in sub.iterrows())
    out(f'  {g:<18} {items}')

PATH_REPORT.write_text('\n'.join(lines))
print(f'\nReport written to {PATH_REPORT}')


NOTEBOOK 08: TEMPORAL ANALYSIS REPORT
  bin width                 : 5 years
  range covered             : 1915 to 2015 (21 bins)
  films per bin (min/max)   : 248 / 7087
  narrative pairs tracked   : 28,066
  per-(element, type, genre, bin) cells after filter: 815,444
  per-(element, type, genre) lifecycle rows: 143,691
  ever-stock rate           : 10.6%

HEADLINE: CORPUS-WIDE SURVIVAL CURVE
Given an element is at z_ensemble ≥ 1.0 in some genre at time τ,
what is the probability it is still above the threshold at time τ + lag?

   lag (years)  P(stock | stock at τ)      n_pairs
             5                  0.568       44,035
            10                  0.547       40,677
            15                  0.538       36,433
            20                  0.535       32,540
            30                  0.517       26,161
            50                  0.492       16,373

PER-GENRE SURVIVAL AT LAG = 20 YEARS
(higher = genre's conventions are more durable)
  Drama              P

## Outputs produced

- **`stockiness_by_bin.parquet`** — per-`(element, type, genre, bin)` long table with all four measures, within-(bin, genre) z-scores, and the ensemble. The temporal analogue of `stockiness_long.parquet`. One row per cell.
- **`element_lifecycle.parquet`** — per-`(element, type, genre)` lifecycle summary: first bin, last bin, peak bin, peak ensemble, first stock crossing, last stock crossing, stock duration in bins, ever-stock flag, count of bins present. Smoothed via LOWESS frac=0.3.
- **`survival_curves.parquet`** — survival probabilities at lags 5, 10, 15, 20, 30, 50 years, per genre and corpus-wide. Each row: `lag_years`, `genre`, `n_pairs`, `n_survived`, `survival_probability`.
- **`change_points.parquet`** — per-`(genre, from_bin, to_bin)` Jaccard similarity of consecutive-bin top-K, with a boolean `is_change_point` flag for bins in the bottom 10% of within-genre Jaccards.
- **`temporal_report.txt`** — headline corpus-wide survival curve, per-genre survival at lag=20, persistent conventions per genre, emerging conventions (first stock 1985+), fading conventions (peak <1990 AND last stock <2000), and per-genre vocabulary change-points.

## How to read the temporal results

The doctrinal target is the Restatement §18 "customary" descriptor. The notebook delivers three complementary lenses on it.

The **survival curve** quantifies the typical lifespan of a genre convention. If the corpus-wide P(stock at τ + 20 | stock at τ) is, say, 0.40, then roughly forty percent of conventions remain conventional twenty years after first becoming so. High survival probabilities at long lags mean a genre's vocabulary is durable; low probabilities mean conventions turn over quickly. Per-genre survival probabilities surface genre-specific durability patterns.

The **element lifecycle** identifies specific conventions as customary, emerging, or fading. A convention that has been stock continuously for fifty years is customary in the strict sense — it has survived several generations of writers and audiences and is embedded in the genre's vocabulary. An emerging convention (first stock in the last decade) is not yet customary; the doctrine's "customary" descriptor cuts against the strongest application of scènes à faire to such elements. A fading convention (stock historically but no longer) raises the question of whether it remains scènes à faire of the genre as currently practiced, which has implications for cases decided under contemporary doctrine.

The **vocabulary change-points** identify the bins in which a genre's conventional vocabulary turned over substantially. These are the doctrinally interesting decade or half-decade in which a genre redefined itself. Western's change-points likely cluster around the post-1970s revisionist period; Horror's may cluster around the slasher emergence in the late 1970s and the post-2000 supernatural turn. The change-point bins anchor the historical argument the paper can make about how conventions evolve.

## Known limitations and caveats

- **Bin width is five years.** Finer bins (one or two years) would provide higher resolution but produce noisier per-bin counts in small genres. Five-year bins are the methods-appendix convention.
- **Smoothing dampens single-bin spikes.** LOWESS at frac=0.3 attenuates noise but can also blunt genuine sharp transitions. The raw per-bin ensemble values are saved in `stockiness_by_bin.parquet` for downstream analyses that need unsmoothed input.
- **Films before 1900 are excluded.** The 1895-1899 cohort is too small to support per-bin z-scoring stably; about a hundred films across the entire range are dropped.
- **Pre-1920 bins have small populations.** A bin with ~100 films can produce z-scores that bounce between values depending on which films happened to be released. The `MIN_FILMS_PER_BIN_TOTAL=100` filter excludes the worst sparse bins, but interpretation of 1900-1920 results should remain cautious.
- **Multi-label genres are not partitioned.** A film tagged Drama and Crime contributes to both genres' bin counts. This is consistent with the rest of the pipeline.
- **The stock threshold is a single value (z=1).** The doctrinal threshold for "customary" is not specified by the Restatement; z=1 corresponds to roughly the top 16% of within-bin elements. Sensitivity to threshold can be tested by recomputing the lifecycle and survival statistics at z=0.5 or z=1.5 from the saved per-bin table.

## What feeds where

- **Notebook 06 (convergent validation)** can use the temporal lifecycle to test whether expert-curated or judicially-identified "customary" elements have long stock durations in the framework.
- **Notebook 07 (court case validation)** can use the change-point bins to date judicial identifications: a court opinion from 1985 identifying an element as scènes à faire is more credibly testing the framework against the 1985 cohort's stockiness scores than against the corpus-pooled ones.
- **Notebook 09 (deliverables)** can lift the headline survival curve and the per-genre change-point table as paper figures.